# HFACS LoRA Fine-Tuning
Fine-tunes Llama 3.1 8B on your labeled accident dataset using PEFT/LoRA.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU** (free) or A100 (Colab Pro)
2. Upload your CSV to Google Drive
3. Fill in the CONFIG cell below

## Step 1 — Install dependencies

In [ ]:
!pip install -q peft transformers trl accelerate bitsandbytes datasets huggingface_hub pyyaml scikit-learn

## Step 2 — Mount Google Drive & set config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CONFIG ── fill these in before running ──────────────────────────────────

# Path to your CSV on Google Drive
CSV_PATH = "/content/drive/MyDrive/hfacs_finetune/accidents_labeled.csv"

# Column names in your CSV
NARRATIVE_COLUMN = "narrative"   # column with the accident narrative text
CLASS_COLUMN     = "hfacs_class" # column with the ground-truth HFACS label

# Prompt style to use: 'io', 'io_expanded', or 'cot'
# Must match one of the YAML files in your prompts/ folder
# Upload the chosen YAML to Google Drive alongside the CSV
PROMPT_YAML_PATH = "/content/drive/MyDrive/hfacs_finetune/io_expanded.yaml"

# The prompt key inside the YAML to use as the system/user template
# e.g. 'decision_error', 'supervisory_factors', etc.
# Set to None to build a generic classification prompt instead
PROMPT_KEY = None

# All possible class labels in your CLASS_COLUMN
CLASSES = [
    "decision_error",
    "skill_based_errors",
    "perceptual_error",
    "routine_violation",
    "exceptional_violation",
    "inadequate_supervision",
    "planned_inappropriate_operations",
    "failure_to_correct_known_problems",
    "supervisory_violation",
    "physical_environment_factors",
    "tools_and_technology_issues",
    "operational_process_failures",
    "communication_coordination_planning_failures",
    "fit_for_duty",
    "mental_problems",
    "physiological_state",
    "physical_mental_limitations",
]

# HuggingFace token (needed to download Llama — get it from huggingface.co/settings/tokens)
HF_TOKEN = "hf_YOUR_TOKEN_HERE"

# Where to save the trained adapter (on Google Drive so it persists)
ADAPTER_SAVE_PATH = "/content/drive/MyDrive/hfacs_finetune/hfacs_lora_adapter"

# Training split: fraction of data used for training (rest = validation)
TRAIN_SPLIT = 0.85

# LoRA hyperparameters
LORA_R          = 16
LORA_ALPHA      = 16
LORA_DROPOUT    = 0.05

# Training hyperparameters
NUM_EPOCHS      = 3
BATCH_SIZE      = 2     # increase to 4 if using A100
GRAD_ACCUM      = 4     # effective batch = BATCH_SIZE * GRAD_ACCUM
LEARNING_RATE   = 2e-4
MAX_SEQ_LEN     = 1024
# ────────────────────────────────────────────────────────────────────────────

## Step 3 — Load and preview dataset

In [ ]:
import pandas as pd
import yaml

df = pd.read_csv(CSV_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nClass distribution:")
print(df[CLASS_COLUMN].value_counts())
df[[NARRATIVE_COLUMN, CLASS_COLUMN]].head(3)

## Step 4 — Format training samples

In [ ]:
# Load prompt template from YAML if provided
prompt_template = None
if PROMPT_KEY:
    with open(PROMPT_YAML_PATH, "r", encoding="utf-8") as f:
        prompts = yaml.safe_load(f)
    prompt_template = prompts.get(PROMPT_KEY, "")
    print(f"Using prompt key: '{PROMPT_KEY}'")
    print(prompt_template[:300])

CLASSES_STR = ", ".join(CLASSES)

SYSTEM_PROMPT = (
    "You are an expert in aviation safety and the HFACS (Human Factors Analysis "
    "and Classification System) framework. Given an accident narrative, classify "
    f"it into exactly one of the following categories: {CLASSES_STR}. "
    "Respond with only the category name, nothing else."
)

def build_user_message(narrative: str) -> str:
    if prompt_template:
        # Use the existing YAML prompt — replace {context} with the narrative
        return prompt_template.replace("{context}", narrative)
    # Generic fallback
    return f"Classify this accident narrative:\n\n{narrative}"


def format_sample(narrative: str, label: str) -> str:
    """Format one training sample in Llama 3 chat template."""
    return (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{build_user_message(narrative)}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
        f"{label}"
        "<|eot_id|>"
    )


# Build formatted dataset
df = df.dropna(subset=[NARRATIVE_COLUMN, CLASS_COLUMN])
df["text"] = df.apply(
    lambda r: format_sample(str(r[NARRATIVE_COLUMN]).strip(), str(r[CLASS_COLUMN]).strip()),
    axis=1,
)

print(f"\nTotal samples after cleaning: {len(df)}")
print("\nSample formatted text:")
print(df["text"].iloc[0][:600])

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df, test_size=1 - TRAIN_SPLIT, random_state=42, stratify=df[CLASS_COLUMN]
)

train_dataset = Dataset.from_pandas(train_df[["text"]].reset_index(drop=True))
val_dataset   = Dataset.from_pandas(val_df[["text", NARRATIVE_COLUMN, CLASS_COLUMN]].reset_index(drop=True))

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")

## Step 5 — Load base model and apply LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from huggingface_hub import login

login(token=HF_TOKEN)

BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# 4-bit quantization — required to fit 8B model in Colab VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded. dtype: {model.dtype}")

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=[
        "q_proj", "v_proj", "k_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## Step 6 — Train

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/checkpoints",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",         # save checkpoint after each epoch
    save_total_limit=2,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=training_args,
)

print("Starting training...")
trainer.train()
print("Training complete.")

## Step 7 — Save adapter to Google Drive

In [ ]:
import os
os.makedirs(ADAPTER_SAVE_PATH, exist_ok=True)

model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)

print(f"Adapter saved to: {ADAPTER_SAVE_PATH}")
print(os.listdir(ADAPTER_SAVE_PATH))

## Step 8 — Evaluate accuracy on validation set
Matches the evaluation logic in `src/llm/classify.py`

In [ ]:
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

model.eval()

def predict(narrative: str) -> str:
    """Run inference and return the predicted class label."""
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM_PROMPT}"
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"{build_user_message(narrative)}"
        "<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,   # class label is short
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated.strip().lower()


# Normalize predicted label to closest known class
_label_lookup = {c.lower(): c for c in CLASSES}

def normalize_pred(raw: str) -> str:
    return _label_lookup.get(raw, raw)


print(f"Running inference on {len(val_df)} validation samples...")
val_df = val_df.copy()
val_df["y_pred_raw"] = [
    predict(str(n)) for n in val_df[NARRATIVE_COLUMN]
]
val_df["y_pred"] = val_df["y_pred_raw"].apply(normalize_pred)
val_df["y_true"] = val_df[CLASS_COLUMN].str.strip().str.lower().map(_label_lookup)

valid_mask = val_df["y_pred"].isin(CLASSES) & val_df["y_true"].isin(CLASSES)
invalid = val_df[~valid_mask]
evaluated = val_df[valid_mask]

print(f"\nTotal val rows: {len(val_df)}")
if not invalid.empty:
    print(f"Excluded (unparseable predictions): {len(invalid)}")
    print(invalid[[NARRATIVE_COLUMN, "y_pred_raw"]].head())

print(f"Evaluated rows: {len(evaluated)}")
print(f"\nAccuracy: {accuracy_score(evaluated['y_true'], evaluated['y_pred']):.4f}")
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(evaluated["y_true"], evaluated["y_pred"], labels=CLASSES)
cm_df = pd.DataFrame(
    cm,
    index=[f"True: {c}" for c in CLASSES],
    columns=[f"Pred: {c}" for c in CLASSES],
)
print(cm_df.to_string())
print("\n--- Classification Report ---")
print(classification_report(evaluated["y_true"], evaluated["y_pred"], labels=CLASSES, zero_division=0))

## Step 9 — (Optional) Run on new unlabeled data

In [ ]:
NEW_DATA_CSV = "/content/drive/MyDrive/hfacs_finetune/new_accidents.csv"
OUTPUT_CSV   = "/content/drive/MyDrive/hfacs_finetune/new_accidents_classified.csv"

new_df = pd.read_csv(NEW_DATA_CSV)
print(f"Classifying {len(new_df)} new accidents...")

from tqdm import tqdm
tqdm.pandas()

new_df["predicted_hfacs_class"] = new_df[NARRATIVE_COLUMN].progress_apply(
    lambda n: normalize_pred(predict(str(n)))
)

new_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved to: {OUTPUT_CSV}")
print(new_df["predicted_hfacs_class"].value_counts())

## Step 10 — (Optional) Load saved adapter in a future session

In [ ]:
# Run this cell in a new Colab session to reload the trained adapter
# without retraining.

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_SAVE_PATH)
model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
model.eval()
print("Adapter loaded successfully.")